# 🤖 Agent Tool-Calling Loop — Guided Demo

A runnable walkthrough of a DevRev-flavored **ReAct agent**:
query → pick a tool → observe → loop → answer, with a **max-iteration guard**, **confirmation gate**,
**memoization**, **retry**, **fallback**, **disambiguation**, and **observability** — built both
**from scratch** and with **LangGraph**. The agent code lives in this project's `src/` package;
this notebook drives it step by step.

> **Context:** DevRev Technical Round · Section 3 — the prep note's "tool-calling loop, coded from scratch".

> No API key needed: the "brain" is a deterministic rule-based router that mimics an LLM's tool
> selection, so every cell below is reproducible offline.

## Learning Objectives
In this notebook, you will learn:
1. **The tool registry** - describe each tool by name, description and a `destructive` flag
2. **The ReAct loop** - run think → act → observe until a final answer or the max-iteration guard
3. **The confirmation gate** - pause before a destructive tool and complete the request once approved
4. **Robustness** - retry transient failures, memoize repeats, fall back on permanent failures, disambiguate ties
5. **State and observability** - carry a `Session` across turns and read its full tool-call trace
6. **The same loop in LangGraph** - a two-node `StateGraph` with a conditional edge

## Prerequisites
- Python basics; no earlier notebook is needed
- Dependencies installed with `uv sync` in this project's folder
- No API key: everything here runs offline (this project's `.env` is loaded but not used)


---
## 📦 Step 1: Environment Setup

The setup cell loads this project's `.env`, finds the project root (the folder that contains `src/`)
and imports the agent modules from it.


In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: load .env, make the src/ package importable
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads this project's .env (no key is needed in this notebook)
load_dotenv(find_dotenv(usecwd=True))

import sys, os
# Find the project root (the folder that contains `src/`), whether we launched from
# notebooks/ or from the project root.
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "src")):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

from src.tools import TicketStore, build_registry
from src.brain import RuleBasedBrain
from src.scratch_agent import Agent, Session
from src.robustness import (always_approve, deny_destructive, execute_tool,
                            with_fallback, disambiguate)
from src.observability import ToolCallLogger
print("✅ Environment ready — project root:", _root)

---
## 🧰 Part 1: The Tool Registry

A **tool** = name + description (what the model reads to choose it) + the function + a
`destructive` flag. The registry is what you'd hand an LLM so it knows what it can call.

In [ ]:
# ============================================================================
# TOOL REGISTRY: list every tool the agent can call
# ============================================================================
store = TicketStore()
registry = build_registry(store)
for spec in registry.specs():
    flag = "  ⚠️ destructive" if spec["destructive"] else ""
    print(f"- {spec['name']:<15} {spec['description']}{flag}")

---
## 🔄 Part 2: The ReAct Loop (From Scratch) — a Single-Step Query

`Agent.run` loops: **think** (brain picks a tool or finishes) → **act** (execute) →
**observe** → repeat, until a final answer or the max-iteration guard.

In [ ]:
# ============================================================================
# REACT LOOP: one query, one tool call, one answer
# ============================================================================
agent = Agent(registry, confirm=always_approve)   # auto-approve for the happy path
sess = Session()
result = agent.run("find tickets about auth", sess)
print("ANSWER:", result.answer)
print("iterations:", result.iterations)
print("\nTRACE:")
print(sess.logger.trace())

---
## 🛑 Part 3: Multi-Step + the Confirmation Gate

"close the ticket about login" needs **two** tool calls: `search_tickets` then
`close_ticket`. Because `close_ticket` is **destructive**, the agent pauses for
confirmation. First we deny (nothing happens), then we approve.

In [ ]:
# ============================================================================
# CONFIRMATION GATE (DENY): the destructive step is blocked
# ============================================================================
# 3a. Default policy DENIES destructive actions -> the agent pauses.
store_a = TicketStore(); reg_a = build_registry(store_a)
blocked = Agent(reg_a, confirm=deny_destructive).run("close the ticket about login")
print("BLOCKED:", blocked.answer)
print("blocked_on:", blocked.blocked_on)
print("ticket still open? ->", store_a.tickets["TKT-3"]["status"])

In [ ]:
# ============================================================================
# CONFIRMATION GATE (APPROVE): the same request completes
# ============================================================================
# 3b. With approval, the same request completes.
store_b = TicketStore(); reg_b = build_registry(store_b); sess_b = Session()
done = Agent(reg_b, confirm=always_approve).run("close the ticket about login", sess_b)
print("ANSWER:", done.answer, "| iterations:", done.iterations)
print("ticket status now ->", store_b.tickets["TKT-3"]["status"])
print("\nTRACE:")
print(sess_b.logger.trace())

---
## 🛡️ Part 4: Robustness

Four helpers from `src/robustness.py` turn the toy loop into something you could ship:
retry for transient failures, memoization for repeated calls, fallback for permanent failures,
and a deterministic tie-break between two plausible tools.


### 4.1 🔁 Retry on a Flaky Tool

A transient failure (network blip / 503) should be **retried**, not surfaced. Here
`search_tickets` fails its first two calls, then succeeds; the executor retries.

In [ ]:
# ============================================================================
# RETRY: a tool that fails twice, then succeeds
# ============================================================================
log = ToolCallLogger()
flaky_reg = build_registry(TicketStore(), flaky_search=True)   # fails twice, then works
res = execute_tool(flaky_reg.get("search_tickets"), {"query": "auth"}, logger=log, retries=3)
print("result:", [h["id"] for h in res])
print("retries:", log.count("retry"), "| final ok:", log.count("ok"))
print("\nTRACE:")
print(log.trace())

### 4.2 ⚡ Memoization

An identical `(tool, args)` call within a session is served from cache instead of
re-hitting the backend. Watch the second call become a `cache_hit`.

In [ ]:
# ============================================================================
# MEMOIZATION: the second identical call is a cache hit
# ============================================================================
log = ToolCallLogger(); memo = {}
for _ in range(2):
    execute_tool(registry.get("search_tickets"), {"query": "billing"}, logger=log, memo=memo)
print("ok:", log.count("ok"), "| cache_hit:", log.count("cache_hit"))
print(log.trace())

### 4.3 🔀 Fallback

If a permanent failure happens (`get_ticket` on a missing id), fall back to an
alternative tool (`search_tickets`), optionally re-mapping the arguments.

In [ ]:
# ============================================================================
# FALLBACK: a permanent failure reroutes to another tool
# ============================================================================
log = ToolCallLogger()
out = with_fallback(
    registry.get("get_ticket"), registry.get("search_tickets"),
    {"ticket_id": "TKT-999"},                 # doesn't exist -> permanent failure
    adapt=lambda a: {"query": "login"},        # re-map args for the fallback tool
    logger=log,
)
print("fallback result:", [h["id"] for h in out])
print(log.trace())

### 4.4 ⚖️ Disambiguation

When two tools could plausibly answer the same query, choose deterministically and
**never silently run a destructive tool on a tie** — safe tools win.

In [ ]:
# ============================================================================
# DISAMBIGUATION: a safe tool wins a tie
# ============================================================================
candidates = [registry.get("search_tickets"), registry.get("close_ticket")]
picked = disambiguate(candidates, "find and close tickets about auth")
print("candidates:", [t.name for t in candidates])
print("picked:", picked.name, "(non-destructive preferred)")

---
## 🧠 Part 5: State and Observability

A `Session` is what survives between turns; its logger is the trace you would ship to a tracing backend.


### 5.1 💬 State Across Conversation Turns

A `Session` persists **history + memo + logger** across turns. A repeated search in a
later turn is a free cache hit.

In [ ]:
# ============================================================================
# MULTI-TURN STATE: a repeated search in a later turn is cached
# ============================================================================
conv = Session()
a = Agent(registry, confirm=always_approve)
print(a.run("find tickets about auth", conv).answer)
print(a.run("find tickets about auth", conv).answer)   # identical -> cached
print("\ncache hits this session:", conv.logger.count("cache_hit"))

### 5.2 🔍 The Full Session Trace

Everything the agent did, in order: tool, status (ok/retry/cache_hit/blocked/error),
latency, args → result. This is what you'd ship to LangSmith / OpenTelemetry.

In [ ]:
# ============================================================================
# OBSERVABILITY: print everything the session did
# ============================================================================
print(conv.logger.trace())

---
## 🕸️ Part 6: The Same Loop With LangGraph

Two nodes — `agent` (think) and `tools` (act) — with a conditional edge that loops while
there are tool calls and we're under the iteration budget. The rule-based brain emits an
`AIMessage` with `tool_calls` (the exact shape a real LLM with `bind_tools` returns), so
this runs with no API key.

In [ ]:
# ============================================================================
# LANGGRAPH: the same tools and brain as a two-node StateGraph
# ============================================================================
from src.langgraph_agent import build_graph, run_query
store_lg = TicketStore(); reg_lg = build_registry(store_lg); sess_lg = Session()
app = build_graph(reg_lg, sess_lg, confirm=always_approve)

print(run_query(app, "close the ticket about billing"))
print(run_query(app, "find tickets about auth"))
print("billing ticket status ->", store_lg.tickets["TKT-2"]["status"])

### 6.1 🗺️ The Graph LangGraph Compiled

```mermaid
graph TD;
    __start__ --> agent;
    agent -.->|has tool calls & under budget| tools;
    agent -.->|final answer / out of budget| __end__;
    tools --> agent;
```

In [ ]:
# ============================================================================
# GRAPH SOURCE: LangGraph prints its own Mermaid diagram
# ============================================================================
print(app.get_graph().draw_mermaid())

---
## 📝 Summary

In this notebook, we learned:

### 1. The Tool Registry
- A tool is a name, a description the model reads, the function, and a `destructive` flag.

### 2. The ReAct Loop
- Think → act → observe → repeat, with a **max-iteration guard** so it never runs forever.
- The brain (an LLM or, here, rules) maps intent to the right tool from the registry.

### 3. The Confirmation Gate
- Destructive tools are blocked by default; the same request completes once approved.

### 4. Robustness
- Retry transient failures, fall back on permanent ones, escalate if neither works.
- **Memoize** repeats; **disambiguate** ties toward safe tools.

### 5. State and Observability
- A `Session` carries history, cache and a trace across turns.
- **Latency:** run **independent** tool calls in parallel (latency = max, not sum), keep **dependent**
  steps sequential, and coordinate parallel calls through one rate limiter.

### 6. The Same Loop in LangGraph
- Two nodes (`agent`, `tools`) and one conditional edge express the same loop; the rule-based brain
  emits `AIMessage.tool_calls`, the exact shape a real LLM returns.

### Next Steps
- **`robust_langgraph_tool_calling_agent.ipynb`**: keeps the brain rule-based but declares the robustness
  policy (destructive, fallback, priority, retry budget) on native LangChain `@tool` objects.
